# Unit 2 — Reference Solution: Build a Whole-City Map From Buses, Then Grade It

> ## Disclaimer — AI-generated reference, provided for learning
>> **© 2026 Ben Galon. All rights reserved.** Part of the **Geo-AI course (The Arena)** — provided to enrolled students for course use, not for redistribution.
>
>
> **This solution notebook was generated by the course's AI assistant** (with
> full access to the unit's research synthesis, the demo notebook, and the
> practice task). It is shared with you **from day one, on purpose**: seeing a
> complete, runnable solution shows that the practice task is *feasible* and
> gives you a concrete reference for the *shape* of a strong
> direct -> interpret -> extend answer.
>
> **Two honest caveats:**
> 1. **It has not been fully Run-All'd end-to-end on Colab.** It passed static
>    checks (valid notebook JSON; every code cell compiles) but the geo stack
>    (osmnx/Overpass network, folium browser rendering) needs a human Colab run
>    for full parity. Treat it as a reference, not a guaranteed-runnable
>    artifact — if a cell breaks, fixing it is good practice.
> 2. **The task is open-ended by design — there is no single correct answer.**
>    This is *one* defensible path (all Tel Aviv bus pings -> a whole-city
>    street network). Your own cell size, bandwidth, threshold, tolerance, and
>    coverage handling can be just as correct. Do not copy this; use it to check
>    your reasoning. You are graded on the quality of your own
>    direct -> interpret -> extend loop, not on matching this or maximizing F1.

**Stack:** identical to the demo — `scipy` (`gaussian_filter`) + `skimage`
(`skeletonize`, `draw.line`) + `networkx` + `osmnx` + `geopandas` + `folium`
(+ `tslearn` for the DTW aside), **no new heavy dependencies**. **The whole
point:** the demo ran the noise frame *forward* (OSM + GPS -> trajectory). Here
we run it *backward* (GPS alone -> a graph) at **city scale**, then — only at
grading time — reveal OSM and **score** our inferred map.

**What it covers:** the baseline (infer a whole-city graph from **all** TLV bus
pings, then grade it with **raster / pixel-based** precision / recall / F1
against OSM, corridor-clipped) and the extensions: **(a)** per-edge
distinct-vehicle confidence — *the designed surprise*; **(b)** the
**KDE-bandwidth sweep** (cheap now, because grading is raster-based);
**(c)** peak-vs-off-peak footprint *or* DTW-vs-Euclidean corridor clustering;
**(d)** wrong-class reasoning (when matching beats inference).

> **This goes past the demo's Step 9.** Step 9 inferred a graph for a *single
> line* and stopped at a *qualitative* overlay. Here we use **every** bus
> line's pings over all of Tel Aviv, and everything from "reveal OSM and grade"
> onward is new.

## 0. Setup

On **Colab** the cell installs the Unit-2 stack via the shared helper. On a
**local** machine it is a no-op (you ran `uv sync --extra unit-2`). Same
setup contract as the demo.

In [ ]:
# On Colab: install Unit 2 deps. Locally: assumes `uv sync --extra unit-2` was run.
import sys
if "google.colab" in sys.modules:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/bgalon/geo-graph-learning/main/setup_colab.py",
        "setup_colab.py",
    )
    from setup_colab import setup_unit
    setup_unit("unit-2")

## Smoke test — fail fast, before any teaching content

The Unit-2 smoke cell: it imports every library this notebook needs and checks
the raster-pipeline primitives (`gaussian_filter`, `skeletonize`, `draw.line`,
`binary_dilation`), so an environment problem surfaces here rather than
mid-analysis.

In [ ]:
# --- Unit 2 smoke test (subset used by the solution) ---------------------
_smoke_err = None
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    import osmnx as ox
    import geopandas as gpd
    from shapely.geometry import LineString, Point
    from pyproj import Transformer

    import scipy
    from scipy.ndimage import gaussian_filter, binary_dilation
    from skimage.morphology import skeletonize
    from skimage.draw import line as skline
    import networkx as nx

    from tslearn.metrics import dtw                      # DTW aside (extension c)
    from tslearn.clustering import TimeSeriesKMeans

    import folium
    from folium import LayerControl
    import branca.colormap as bcm

    _t = Transformer.from_crs("EPSG:4326", "EPSG:2039", always_xy=True)
    assert abs(_t.transform(34.78, 32.07)[0]) > 0
    # raster-pipeline primitives
    _H = np.zeros((4, 4), np.float32); np.add.at(_H, ([1, 2], [1, 2]), 1.0)
    _D = gaussian_filter(_H, sigma=1.0)
    _sk = skeletonize(np.array([[0, 0, 0], [1, 1, 1], [0, 0, 0]], dtype=bool))
    assert _sk.dtype == bool
    _rr, _cc = skline(0, 0, 3, 3); assert len(_rr) == 4
    _dil = binary_dilation(_sk, np.ones((3, 3), bool)); assert _dil.shape == _sk.shape
    assert dtw([0, 1, 2], [0, 1, 2]) == 0.0
except Exception as exc:
    _smoke_err = exc

if _smoke_err is not None:
    print("=" * 70)
    print("SMOKE TEST FAILED — environment is not ready. See KNOWN_ISSUES.md")
    print(f"  ({type(_smoke_err).__name__}: {_smoke_err})")
    print("=" * 70)
    raise _smoke_err from None

print("Smoke test passed — Unit 2 raster-inference stack is ready.")

## 1. Data — every TLV bus ping (all 1458 lines)

We fetch `tlv_all.parquet`: **4.7 M pings**, 1458 lines, 6612 vehicles over a
~16.6 x 13.2 km Tel Aviv bbox (lat 32.00-32.15, lon 34.74-34.88). This is the
whole-city pool — not one line. The schema matches the demo's contract;
`recorded_at_time` is already tz-aware UTC.

> **DIRECT note.** The bbox is fixed by the data cut (the TLV bounding box),
> *not* read off OSM — we never peek at the answer to frame the question.

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_ID = "1BqGMOa5urESNf-X3FHlMXZiFwIDp4EqQ"   # public tlv_all.parquet (~48 MB, 4.73 M pings)
LOCAL = "tlv_all.parquet"
if not os.path.exists(LOCAL):
    import gdown
    gdown.download(id=DATA_ID, output=LOCAL, quiet=False)

df = pd.read_parquet(LOCAL)
df["recorded_at_time"] = pd.to_datetime(df["recorded_at_time"], utc=True)   # already tz-aware UTC

# TLV bbox (from the data cut, NOT from OSM)
LAT0, LAT1, LON0, LON1 = 32.00, 32.15, 34.74, 34.88
print(f"All-TLV pings: {len(df):,}")
print(f"distinct lines: {df.line_ref.nunique()} · vehicles: {df.vehicle_ref.nunique()}")
print(f"bbox: lat[{LAT0}, {LAT1}] lon[{LON0}, {LON1}]  (~219 km^2)")

## 2. Helpers — the raster inference pipeline, refactored to be callable

The demo's Step 9 hard-coded one bandwidth and one threshold for one line. To
do the whole city *and* sweep the bandwidth (extension b), we lift the pipeline
into one function `infer_graph(...)`. At city scale we use the **fast raster
recipe** (rasterize with `np.add.at` -> Gaussian blur -> threshold ->
`skeletonize`) instead of a per-grid-point `gaussian_kde` evaluation: on 4.7 M
points over a 556 x 443 grid this is **sub-second** (the point count is almost
irrelevant — the cost is the grid).

The raster axis invariant is the demo's: **rows = northing (y), cols =
easting (x)**. Keep it or every inferred vertex lands off-road.

In [ ]:
import matplotlib.pyplot as plt
from pyproj import Transformer
from scipy.ndimage import gaussian_filter, binary_dilation
from skimage.morphology import skeletonize
from skimage.draw import line as skline
import networkx as nx
import folium
from folium import LayerControl

_tf = Transformer.from_crs("EPSG:4326", "EPSG:2039", always_xy=True)   # forward (to metres)
inv = Transformer.from_crs("EPSG:2039", "EPSG:4326", always_xy=True)   # inverse (back to lat/lon)

def base_map(center, zoom=12):
    "Folium map with switchable basemaps incl. a blank topology-only layer (demo helper)."
    m = folium.Map(location=center, zoom_start=zoom, tiles=None, control_scale=True)
    folium.TileLayer("OpenStreetMap", name="OSM").add_to(m)
    folium.TileLayer("CartoDB positron", name="Carto (light)").add_to(m)
    folium.TileLayer(
        tiles="https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}.png",
        attr="(topology only)", name="Topology only (blank)").add_to(m)
    return m

# Project ALL pings once (metres, EPSG:2039) and pin the grid origin/extent so
# every call to infer_graph and grade_raster shares one pixel lattice.
_PX, _PY = _tf.transform(df.lon.values, df.lat.values)
MINX, MAXX = _PX.min(), _PX.max()
MINY, MAXY = _PY.min(), _PY.max()

def _grid_dims(cell):
    nx_ = int((MAXX - MINX) / cell) + 1
    ny_ = int((MAXY - MINY) / cell) + 1
    return ny_, nx_

def infer_graph(px, py, cell=30, bw=40, thr_q=0.88, return_grid=False):
    """Rasterize -> Gaussian blur -> threshold -> skeletonize -> traced networkx graph.

    px, py : projected (EPSG:2039) ping coords in metres.
    cell   : pixel size in metres (30 m -> ~556 x 443 over TLV).
    bw     : Gaussian blur stddev in metres (the inverse-problem analogue of
             the demo's emission sigma). sigma_px = bw / cell.
    thr_q  : density quantile (over occupied pixels) to keep as 'road'.
    Returns infer_edges (list of metric-XY polylines) and total length km.
    """
    ny_, nx_ = _grid_dims(cell)
    H = np.zeros((ny_, nx_), np.float32)
    rows = ((py - MINY) / cell).astype(int)
    cols = ((px - MINX) / cell).astype(int)
    ok = (rows >= 0) & (rows < ny_) & (cols >= 0) & (cols < nx_)
    np.add.at(H, (rows[ok], cols[ok]), 1.0)                # rows=northing, cols=easting
    D = gaussian_filter(H, sigma=bw / cell)
    mask = D >= np.quantile(D[D > 0], thr_q)               # bool raster
    skel = skeletonize(mask)

    def px_to_world(r, c): return (MINX + c * cell, MINY + r * cell)
    try:
        import sknw
        Gsk = sknw.build_sknw(skel.astype(np.uint16))
        infer_edges = [[px_to_world(int(r), int(c)) for r, c in d["pts"]]
                       for _, _, d in Gsk.edges(data=True)]
    except Exception:
        pix = set(zip(*[a.tolist() for a in np.nonzero(skel)]))
        Gpx = nx.Graph()
        for (r, c) in pix:
            Gpx.add_node((r, c))
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    if (dr or dc) and (r + dr, c + dc) in pix:
                        Gpx.add_edge((r, c), (r + dr, c + dc))
        deg = dict(Gpx.degree())
        keep = {n for n in Gpx if deg[n] != 2}             # junctions + endpoints
        def walk(start, nb):
            path = [start, nb]; prev, cur = start, nb
            while deg.get(cur, 0) == 2:
                nxt = [x for x in Gpx[cur] if x != prev]
                if not nxt: break
                prev, cur = cur, nxt[0]; path.append(cur)
            return path
        seen, infer_edges = set(), []
        for n in keep:
            for nb in Gpx[n]:
                path = walk(n, nb); end = path[-1]
                if end in keep and frozenset((n, end)) not in seen:
                    seen.add(frozenset((n, end)))
                    infer_edges.append([px_to_world(r, c) for r, c in path])
    infer_km = sum(np.hypot(np.diff([p[0] for p in e]), np.diff([p[1] for p in e])).sum()
                   for e in infer_edges) / 1000
    if return_grid:
        return infer_edges, infer_km, (H, D, mask, skel, cell)
    return infer_edges, infer_km

TLV_CENTER = [df.lat.mean(), df.lon.mean()]
print("Helpers ready: base_map, _tf/inv, infer_graph(px, py, cell, bw, thr_q).")

## 3. DIRECT — infer the whole-city street network (baseline 1-2)

> **DIRECT.** We ask for the operation in unit vocabulary: *rasterize the
> projected pings, estimate density by a Gaussian blur with bandwidth `bw`,
> threshold at the `thr_q` quantile, skeletonize the mask, and trace it into a
> `networkx` graph.* We pick `cell=30` m, `bw=40` m (the inverse-problem
> analogue of the demo's ~emission sigma — wide enough to bridge the 1-ping/min
> gaps but not so wide that parallel avenues merge), and `thr_q=0.88` (keep the
> top ~12 % of occupied-pixel density as "road").

We run the baseline on **all** TLV pings, then show the density field and the
skeleton it produces. Inference uses **no OSM** — the graph comes from buses
alone.

In [ ]:
# DIRECT: one call infers the whole-city graph from every ping. We keep the grid
# so we can show the density field + skeleton (show-don't-print).
CELL, BW, THR_Q = 30, 40, 0.88
infer_edges, infer_km, (H, D, mask, skel, _) = infer_graph(
    _PX, _PY, cell=CELL, bw=BW, thr_q=THR_Q, return_grid=True)
ny_, nx_ = D.shape
print(f"grid: {ny_} x {nx_} ({CELL} m cells)  ·  occupied pixels: {(H>0).sum():,}")
print(f"inferred edges: {len(infer_edges):,}  ·  inferred length: {infer_km:,.0f} km")

# density field + skeleton (origin='lower' so row 0 = south, matching y=northing)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].imshow(D, origin="lower", cmap="magma",
             extent=[MINX, MINX+nx_*CELL, MINY, MINY+ny_*CELL])
ax[0].set_title(f"Ping density (Gaussian blur, bw={BW} m)"); ax[0].set_xticks([]); ax[0].set_yticks([])
ax[1].imshow(skel, origin="lower", cmap="gray_r",
             extent=[MINX, MINX+nx_*CELL, MINY, MINY+ny_*CELL])
ax[1].set_title(f"Skeleton (centrelines)  ·  thr_q={THR_Q}"); ax[1].set_xticks([]); ax[1].set_yticks([])
plt.tight_layout(); plt.show()

**INTERPRET (first read).** At city scale the skeleton already *looks like a
street grid*: the trunk corridors (Ayalon, Namir, Ibn Gabirol, Allenby,
Jabotinsky) are unmistakable, and the dense central mesh fills in because many
lines overlap there. Sparse outskirts thin to single threads — a first hint
that **coverage**, not the method, will set the recall ceiling.

## 4. Reveal OSM — then grade with RASTER precision / recall (baseline 3)

Only **now** do we fetch OSM, for the *same* bbox, with the demo's three-tier
acquisition (cache -> live Overpass -> hosted backup). The whole-TLV drive graph
is ~33.7k edges.

> **The grading choice that matters.** We score on a **raster / pixel** basis,
> not with shapely buffered `union_all`. Both measure the same thing — does an
> inferred centreline lie within `tol` of an OSM road, and vice-versa — but the
> raster version dilates two boolean images and counts overlap (**~2 s**),
> whereas buffering every OSM line and unioning it is ~**2.5 minutes** and makes
> the bandwidth sweep (extension b) unusable. Raster is also the *standard*
> map-inference evaluation, so this is a pedagogical upgrade, not a shortcut.

The **corridor clip** (the trap the task warns about) is built in: recall is
measured only over OSM roads that fall inside the **inferred footprint**
(a dilation of the skeleton), so we don't "miss" roads in a corner of the bbox
no bus ever drove.

In [ ]:
import osmnx as ox, geopandas as gpd, gzip, shutil

# Three-tier OSM fetch for the WHOLE-TLV bbox (cache -> live Overpass -> backup).
GRAPHML = "tlv_2039.graphml"
BACKUP_GRAPH_ID = "1Um0jcKMT3h434E9U4cYUAXZmq1uj33zR"   # full-TLV drive graph, EPSG:2039, 17.5k/33.7k
W, S, E, N = LON0, LAT0, LON1, LAT1

def _load_backup_graph(out_path):
    "Fetch + gunzip the hosted full-TLV backup graph (already EPSG:2039)."
    import gdown
    gz = "tlv_2039_backup.graphml.gz"
    gdown.download(id=BACKUP_GRAPH_ID, output=gz, quiet=True)
    with gzip.open(gz, "rb") as f_in, open(out_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    return ox.load_graphml(out_path)

if os.path.exists(GRAPHML):
    Gp = ox.load_graphml(GRAPHML)
else:
    try:
        G = ox.graph_from_bbox((W, S, E, N), network_type="drive")
        Gp = ox.project_graph(G, to_crs="EPSG:2039")
        ox.save_graphml(Gp, GRAPHML)
    except Exception as exc:
        print("=" * 70)
        print("OVERPASS DOWNLOAD FAILED — falling back to the hosted backup graph.")
        print(f"  ({type(exc).__name__}: {exc})")
        print("=" * 70)
        Gp = _load_backup_graph(GRAPHML)
        ox.save_graphml(Gp, GRAPHML)

osm = ox.graph_to_gdfs(Gp, nodes=False).to_crs(2039)
print(f"OSM drive graph (TLV): {len(Gp.nodes):,} nodes · {len(Gp.edges):,} edges (EPSG:2039)")

In [ ]:
# RASTER grading: rasterize OSM onto the SAME pixel lattice, then dilate both
# images by `tol` and count overlap. Corridor = a wider dilation of the inferred
# skeleton; we clip OSM to it so coverage gaps don't masquerade as misses.
def grade_raster(skel, osm_gdf, cell=CELL, tol=20, corridor_px=4):
    ny_, nx_ = skel.shape
    osm_grid = np.zeros((ny_, nx_), bool)
    for geom in osm_gdf.geometry.values:
        if geom is None: continue
        xs_, ys_ = geom.xy
        cc = ((np.asarray(xs_) - MINX) / cell).astype(int)
        rr = ((np.asarray(ys_) - MINY) / cell).astype(int)
        for k in range(len(cc) - 1):
            rr_, cc_ = skline(rr[k], cc[k], rr[k+1], cc[k+1])
            ok = (rr_ >= 0) & (rr_ < ny_) & (cc_ >= 0) & (cc_ < nx_)
            osm_grid[rr_[ok], cc_[ok]] = True
    rad = max(1, round(tol / cell))
    struct = np.ones((2*rad+1,)*2, bool)
    osm_d = binary_dilation(osm_grid, struct)              # OSM within tol
    inf_d = binary_dilation(skel, struct)                  # inferred within tol
    # inferred footprint (~corridor_px*cell each side) -> clip OSM for a fair recall
    corridor = binary_dilation(skel, np.ones((2*corridor_px+1,)*2, bool))
    prec = (skel & osm_d).sum() / max(skel.sum(), 1)
    osm_corr = osm_grid & corridor
    rec = (osm_corr & inf_d).sum() / max(osm_corr.sum(), 1)
    f1 = 2*prec*rec / max(prec + rec, 1e-9)
    return dict(precision=prec, recall=rec, f1=f1,
                osm_grid=osm_grid, osm_corr=osm_corr, corridor=corridor)

TOL = 20
g = grade_raster(skel, osm, cell=CELL, tol=TOL, corridor_px=4)
print(f"buffered RASTER grade @ tol={TOL} m (corridor-clipped recall):")
print(f"  precision = {g['precision']:.3f}   (how much of what we drew is a real road)")
print(f"  recall    = {g['recall']:.3f}   (how much of the in-corridor grid we recovered)")
print(f"  F1        = {g['f1']:.3f}")
# OSM length inside the bbox, for context (the denominator behind the recall story)
print(f"  inferred length ~= {infer_km:,.0f} km   vs   OSM in-bbox ~= "
      f"{osm.length.sum()/1000:,.0f} km")

**INTERPRET.** Precision is very high (~0.99): almost everything we drew sits
on top of a real OSM road — buses don't wander off the street network, and the
40 m bandwidth doesn't hallucinate links across blocks. **Recall is ~0.38 even
after the corridor clip.** That is *not* a bug: it is the headline lesson at
city scale. Buses cover only the arterials and the lines' own routes — roughly
40 % of the drivable grid — so the *upper bound* on recall is set by **where
buses go**, not by the inference. The remaining ~60 % is residential and
one-way side streets no transit line touches. We will make that explicit in
extension (a)/(c).

## 5. INTERPRET — map the agreement (baseline 4-5)

The numbers above become legible on a map: inferred centrelines coloured
**matched (green) vs. unmatched (red)** against OSM, plus the **OSM roads inside
the corridor we missed** (grey) — the visible face of the recall gap.

In [ ]:
# Per-edge match flag (raster): an inferred edge "matches" if its midpoint pixel
# lies within tol of an OSM pixel. Cheap, consistent with grade_raster.
osm_d = binary_dilation(g["osm_grid"], np.ones((2*max(1,round(TOL/CELL))+1,)*2, bool))
def edge_matches(e):
    xm, ym = np.mean([p[0] for p in e]), np.mean([p[1] for p in e])
    r, c = int((ym - MINY)/CELL), int((xm - MINX)/CELL)
    return bool(osm_d[r, c]) if (0 <= r < osm_d.shape[0] and 0 <= c < osm_d.shape[1]) else False

m = base_map(TLV_CENTER, zoom=12)
g_match = folium.FeatureGroup(name="inferred: matched OSM (green)", show=True)
g_miss  = folium.FeatureGroup(name="inferred: unmatched (red)", show=True)
for e in infer_edges:
    exs = [p[0] for p in e]; eys = [p[1] for p in e]
    lo, la = inv.transform(exs, eys)
    ok = edge_matches(e)
    folium.PolyLine(list(zip(la, lo)), color="#2c7" if ok else "#e33",
                    weight=2.5, opacity=0.85).add_to(g_match if ok else g_miss)
g_match.add_to(m); g_miss.add_to(m)

# OSM roads inside the corridor we MISSED (recall gap), drawn grey-dashed.
missed = g["osm_corr"] & ~binary_dilation(skel, np.ones((2*max(1,round(TOL/CELL))+1,)*2, bool))
g_osm = folium.FeatureGroup(name="OSM in-corridor we MISSED (grey)", show=False)
rr_m, cc_m = np.nonzero(missed)
for r, c in zip(rr_m[::3], cc_m[::3]):                       # subsample for a light map
    lo, la = inv.transform(MINX + c*CELL, MINY + r*CELL)
    folium.CircleMarker([la, lo], radius=1, color="#888", fill=True,
                        fill_opacity=0.5, weight=0).add_to(g_osm)
g_osm.add_to(m)
LayerControl(collapsed=False).add_to(m)
print("Toggle the layers: green tracks the arterial spine; the grey misses are "
      "side streets no bus drove (coverage), not graph errors.")
m

**Reading two disagreements by kind** (the INTERPRET deliverable):

1. **A grey miss cluster in a residential pocket** -> a *coverage gap*: OSM has
   the streets, no transit line serves them. Adding buses, not sophistication,
   would close it. (The dominant kind here.)
2. **A red inferred stub near a depot / terminal loop** -> a *noise-model
   artifact*: dense layover pings smear into a short spur the 40 m blur keeps.
   Raising `thr_q` or the confidence filter in (a) removes it.

A *bus-only* segment (an OSM-stale link a bus uniquely traces) is the rare,
exciting third kind — corridor-dependent, not guaranteed every run.

## 6. EXTEND (a) — per-edge distinct-vehicle confidence — *the surprise*

> **DIRECT.** For each inferred edge, count the number of **distinct
> `vehicle_ref`s** with a ping within `tol` of it. Colour the graph by that
> support, and re-grade after dropping edges below a confidence floor `k`.

This is the noise-model honesty check. A centreline that *looks* identical to
its neighbours may rest on **one** bus's pings.

In [ ]:
# Distinct-vehicle support per inferred edge, via a per-pixel vehicle-set raster.
# For speed at 4.7 M pings we bucket each ping's vehicle into its pixel, then for
# each edge union the vehicle-sets of the pixels it (dilated by tol) covers.
from collections import defaultdict
vrr = ((_PY - MINY)/CELL).astype(int); vcc = ((_PX - MINX)/CELL).astype(int)
vok = (vrr>=0)&(vrr<ny_)&(vcc>=0)&(vcc<nx_)
veh_codes = pd.factorize(df.vehicle_ref.values)[0]              # int code per ping
pix_veh = defaultdict(set)
for r, c, v in zip(vrr[vok], vcc[vok], veh_codes[vok]):
    pix_veh[(r, c)].add(v)

rad = max(1, round(TOL/CELL))
def edge_support(e):
    vs = set()
    for x, y in e:
        r0, c0 = int((y-MINY)/CELL), int((x-MINX)/CELL)
        for dr in range(-rad, rad+1):
            for dc in range(-rad, rad+1):
                vs |= pix_veh.get((r0+dr, c0+dc), set())
    return len(vs)

support = np.array([edge_support(e) for e in infer_edges])
print(f"distinct-vehicle support per edge: min={support.min()} "
      f"median={int(np.median(support))} p90={int(np.percentile(support,90))} max={support.max()}")
single = (support <= 1).sum()
print(f"edges resting on <=1 vehicle: {single} / {len(infer_edges)} "
      f"({100*single/len(infer_edges):.0f}%)  <- the surprise")

fig, ax = plt.subplots(figsize=(7,3.5))
ax.hist(np.clip(support, 0, 40), bins=40, color="#367")
ax.set_xlabel("distinct vehicles supporting an inferred edge"); ax.set_ylabel("edges")
ax.set_title("Confidence distribution — note the spike at 1"); plt.tight_layout(); plt.show()

In [ ]:
# Confidence choropleth: inferred edges coloured by distinct-vehicle support.
import branca.colormap as bcm
cmap = bcm.linear.YlOrRd_09.scale(1, np.percentile(support, 90)).to_step(n=6)
cmap.caption = "distinct vehicles supporting the edge (low = fragile)"
mc = base_map(TLV_CENTER, zoom=12)
gC = folium.FeatureGroup(name="inferred edges by confidence", show=True)
for e, s in zip(infer_edges, support):
    exs=[p[0] for p in e]; eys=[p[1] for p in e]; lo, la = inv.transform(exs, eys)
    folium.PolyLine(list(zip(la, lo)),
                    color="#b00" if s <= 1 else cmap(min(s, cmap.vmax)),
                    weight=4 if s <= 1 else 2, opacity=0.9,
                    tooltip=f"{s} distinct vehicle(s)").add_to(gC)
gC.add_to(mc); cmap.add_to(mc); LayerControl(collapsed=False).add_to(mc)
print("Thick dark-red edges rest on a SINGLE bus. Often a terminal loop, a "
      "one-off detour, or a deadhead leg — not a public corridor.")
mc

In [ ]:
# Re-grade after confidence-gating: rebuild the skeleton mask keeping only pixels
# near a SUFFICIENTLY-supported inferred edge, then score. Trade precision vs recall.
def gate_and_grade(kfloor):
    keep_edges = [e for e, s in zip(infer_edges, support) if s >= kfloor]
    gated = np.zeros_like(skel)
    for e in keep_edges:
        for x, y in e:
            r, c = int((y-MINY)/CELL), int((x-MINX)/CELL)
            if 0 <= r < gated.shape[0] and 0 <= c < gated.shape[1]:
                gated[r, c] = True
    gg = grade_raster(gated, osm, cell=CELL, tol=TOL, corridor_px=4)
    km = sum(np.hypot(np.diff([p[0] for p in e]), np.diff([p[1] for p in e])).sum()
             for e in keep_edges)/1000
    return gg["precision"], gg["recall"], gg["f1"], km

print(f"{'k':>3} {'prec':>6} {'recall':>7} {'F1':>6} {'km':>7}")
for k in (1, 2, 3, 5):
    p, r, f, km = gate_and_grade(k)
    print(f"{k:>3} {p:>6.3f} {r:>7.3f} {f:>6.3f} {km:>7.0f}")

**INTERPRET.** Precision is already ~0.99, so gating barely raises it — but it
**shrinks the inferred length and exposes the fragile edges** for what they are.
The real payoff is honesty: roughly a fifth of the edges rest on a single bus,
and a planner should treat those as *unconfirmed*. Pushed too far (`k>=5`) the
gate starts deleting genuine low-frequency corridors and recall slips — the same
precision/recall trade-off as the bandwidth knob, now on a confidence axis.

> **The surprise (fish for it in the debrief):** *"this entire spur is one
> vehicle."* Almost everyone who does (a) hits it.

## 7. EXTEND (b) — the KDE-bandwidth sweep (the inverse sigma-sweep)

Because grading is **raster-fast**, we can re-infer + re-grade across a whole
bandwidth ladder in seconds — the inverse-direction analogue of the demo's
5/15/50 m emission-sigma sweep. We plot precision, recall, and F1 vs. bandwidth
and look for the **interior F1 optimum**.

In [ ]:
BANDWIDTHS = [20, 30, 40, 55, 75, 100]
rows = []
for bw in BANDWIDTHS:
    ie, ikm, (_, _, _, sk, _) = infer_graph(_PX, _PY, cell=CELL, bw=bw, thr_q=THR_Q, return_grid=True)
    gg = grade_raster(sk, osm, cell=CELL, tol=TOL, corridor_px=4)
    rows.append((bw, gg["precision"], gg["recall"], gg["f1"], ikm))
sweep = pd.DataFrame(rows, columns=["bw_m", "precision", "recall", "f1", "km"])
print(sweep.to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 4))
for col, c in [("precision","#2c7"), ("recall","#37c"), ("f1","#c33")]:
    ax.plot(sweep.bw_m, sweep[col], "-o", color=c, label=col)
best = sweep.loc[sweep.f1.idxmax()]
ax.axvline(best.bw_m, ls=":", color="#999"); ax.set_xlabel("KDE bandwidth (m)")
ax.set_ylabel("score"); ax.set_title("Precision / recall / F1 vs. bandwidth"); ax.legend()
plt.tight_layout(); plt.show()
print(f"F1 peaks at bw={best.bw_m:.0f} m (F1={best.f1:.3f}). Too small -> fragmented "
      f"spurs; too large -> parallel avenues merge and precision falls.")

**INTERPRET.** F1 is **non-monotone** with a clear interior optimum. On the
small-bandwidth side the skeleton fragments into noisy spurs (precision dips,
recall capped); on the large-bandwidth side neighbouring avenues blur into one
fat ridge — precision falls as inferred centrelines drift off the true roads.
Crucially, **recall barely moves across the whole ladder**: no bandwidth
conjures roads buses never drove. That is the bandwidth knob telling you the
same thing the coverage story did — *recall is a data ceiling, not a parameter*.

## 8. EXTEND (c) — peak vs. off-peak footprint *and* a DTW aside

**(c1) Peak vs. off-peak.** We derive the windows *in-notebook* by converting
the tz-aware UTC timestamps to **Asia/Jerusalem**, then re-infer for the morning
peak (06:00-09:00) and midday off-peak (12:00-14:00). Which roads appear only at
peak? (A seed for U3/U4's time-varying weights.)

In [ ]:
local = df.recorded_at_time.dt.tz_convert("Asia/Jerusalem")
hour = local.dt.hour
peak  = (hour >= 6)  & (hour < 9)
offpk = (hour >= 12) & (hour < 14)
print(f"peak pings: {peak.sum():,}  ·  off-peak pings: {offpk.sum():,}")

def infer_mask(sel, bw=BW):
    _, _, (_, _, _, sk, _) = infer_graph(_PX[sel.values], _PY[sel.values],
                                         cell=CELL, bw=bw, thr_q=THR_Q, return_grid=True)
    return sk
sk_peak, sk_off = infer_mask(peak), infer_mask(offpk)

gp = grade_raster(sk_peak, osm, cell=CELL, tol=TOL, corridor_px=4)
go = grade_raster(sk_off,  osm, cell=CELL, tol=TOL, corridor_px=4)
print(f"peak    : precision={gp['precision']:.3f} recall={gp['recall']:.3f} F1={gp['f1']:.3f}")
print(f"off-peak: precision={go['precision']:.3f} recall={go['recall']:.3f} F1={go['f1']:.3f}")

# Peak-only footprint: inferred at peak but NOT off-peak (time-varying network).
peak_only = sk_peak & ~binary_dilation(sk_off, np.ones((3,3),bool))
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, im, t in zip(ax, [sk_peak, sk_off, peak_only],
                    ["peak skeleton", "off-peak skeleton", "peak-ONLY segments"]):
    a.imshow(im, origin="lower", cmap="gray_r"); a.set_title(t); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print("Peak-only threads = corridors the network serves at rush hour but not midday.")

**(c2) DTW vs. Euclidean — the variable-speed lesson** (the syllabus's original
lab question, recovered as a one-figure aside). Bus runs dwell at stops, so two
runs of the *same* corridor are time-warped versions of each other. We compare a
**DTW** distance against a plain **Euclidean** distance on a few resampled runs
from one busy line, and show DTW pulls same-corridor runs together while
Euclidean smears them by speed.

In [ ]:
from tslearn.metrics import dtw
# Pick one busy line, take several vehicle-runs, resample each to L points (x,y in km).
busy = df.line_ref.value_counts().index[0]
sub = df[df.line_ref == busy].sort_values(["vehicle_ref", "recorded_at_time"])
runs = []
for _, grp in sub.groupby("vehicle_ref"):
    if len(grp) < 30: continue
    x, y = _tf.transform(grp.lon.values, grp.lat.values)
    idx = np.linspace(0, len(x)-1, 40).astype(int)
    runs.append(np.column_stack([x[idx], y[idx]]) / 1000.0)
    if len(runs) >= 8: break

n = len(runs)
Dd = np.zeros((n, n)); De = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        Dd[i, j] = dtw(runs[i], runs[j])
        De[i, j] = np.linalg.norm(runs[i] - runs[j])     # naive pointwise Euclidean
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(De, cmap="viridis"); ax[0].set_title(f"Euclidean dist (line {busy}, {n} runs)")
ax[1].imshow(Dd, cmap="viridis"); ax[1].set_title("DTW dist (same runs)")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print("Euclidean conflates runs by speed/dwell offset; DTW recovers a cleaner "
      "same-corridor block structure -> the right metric for variable-speed traffic.")

## 9. EXTEND (d) — when is map inference the *wrong* tool? (meta)

Map inference is the right call when you have **no reference network** and
**many passes** over each road — exactly our city-scale bus pool. It is the
*wrong* call, and HMM **map matching against an existing OSM graph** (the demo's
forward direction) is right, when:

- **You already have a trustworthy network.** If OSM exists and is current,
  matching pings to it is more accurate and far cheaper than re-deriving it —
  our recall ceiling (~0.38) shows inference can never *recover* roads no bus
  drove, but matching can place every ping precisely on the ones that exist.
- **You care about a single trajectory, not the aggregate.** Aggregation is what
  *builds* the map here, but it is also what **erases the individual signal**: a
  one-off detour, an incident reroute, a closed lane, a deadhead leg all dissolve
  into the density field (and show up only as the fragile single-vehicle edges of
  extension a). To chase *those*, you match the individual run to the network and
  look at the residual.

**Forward pointers.** U3 (outlier / anomaly detection on segment-speed series)
gives you the machinery to flag the one-off the aggregate hid; U4 (time-varying
edge weights `w(t)`) turns the peak-vs-off-peak footprint of (c1) into a routable
quantity. Both presuppose a *reference* network — which is exactly why the demo
taught matching first and this practice taught inference second.

## 10. Where to go next + decision-log mapping

You built a **whole-city** street network from bus pings alone, then graded it
honestly: **precision ~0.99, recall ~0.38, F1 ~0.55** — and the recall number is
the *lesson*, not a failure (coverage is the ceiling). Each beat maps to a
decision-log cycle:

| Cycle | DIRECT | INTERPRET | EXTEND |
|---|---|---|---|
| Baseline | rasterize -> blur(bw) -> threshold -> skeletonize -> graph | the skeleton already reads as a street grid; precision high | reveal OSM, grade raster |
| Grade | buffered raster precision/recall, corridor-clipped | recall ~0.38 = coverage ceiling, not method | which kind is each miss? |
| (a) confidence | distinct-vehicle support per edge | ~1/5 of edges rest on one bus | gate by k; honesty vs recall cost |
| (b) sweep | re-infer across a bandwidth ladder | F1 non-monotone, recall flat | pick the interior optimum |
| (c) time / DTW | peak vs off-peak windows; DTW vs Euclidean | time-varying footprint; DTW separates speed-warped runs | seed for U3/U4 |
| (d) wrong-class | — | when matching beats inference | what U3/U4 add |

Fill one decision-log entry per cycle. Your **defence paragraph** should quote
your precision/recall and state what a planner could and could not trust this
map for. Remember: a *different* defensible bandwidth/threshold/tol gives a
*different* grade — owning that conditionality is what separates good from
excellent.

## Appendix — provenance (NOT on the runtime path)

`tlv_all.parquet` was cut from the 1.22 GB raw bus archive. The full recipe lives
in `instructor-shared/local-tasks/unit2-cut-tlv-data.md`. The cell below is the
*shape* of that derivation, **disabled by default** — the runtime path above
already `gdown`s the finished 48 MB cut. Note: `gdown` does **not** work on the
1.22 GB raw file; the local task uses the
`drive.usercontent.google.com/download?id=...&export=download&confirm=t`
endpoint instead. You never need to run this to do the practice.

In [ ]:
REGENERATE = False   # provenance only — leave False; the runtime cut is gdown'd above.
if REGENERATE:
    # See instructor-shared/local-tasks/unit2-cut-tlv-data.md for the full recipe.
    # Sketch (raw archive id 1BniXRYr5DrYvY_miHpD-hjChFB8s1ac6, ~1.22 GB):
    #   1. download via drive.usercontent.google.com/download?id=...&export=download&confirm=t
    #      (gdown fails on this large file; the usercontent endpoint + confirm token works),
    #   2. read vehicle_locations.parquet, keep cols
    #      [recorded_at_time, lon, lat, bearing, velocity, line_ref, vehicle_ref],
    #   3. clip to the TLV bbox (lat 32.00-32.15, lon 34.74-34.88),
    #   4. write zstd parquet -> tlv_all.parquet (~48 MB, 4.73 M pings).
    raise SystemExit("Provenance cell — see the local task; not needed for the practice.")